# Tools
Tools are defined as Python functions decorated with @tool from LangChain, accompanied by docstrings describing their functionality, which aids the LLM in deciding when to call them.

In [4]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.1-8b-instant")
response = model.invoke("what is ai and ml in short definition")
print(response.content)


Here are short definitions for AI and ML:

**Artificial Intelligence (AI)**:
Artificial Intelligence is a broad field of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making.

**Machine Learning (ML)**:
Machine Learning is a subset of AI that involves training computers to learn from data without being explicitly programmed. It enables machines to automatically improve their performance on a task by analyzing and interpreting data, and to make predictions or decisions based on that data.


In [21]:
#Tools

from langchain.tools import tool 
@tool
def get_weather(location:str)-> str:
    """ get weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])


In [22]:
response = model_with_tools.invoke("What is the weather in delhi")
print(response.text)

In [23]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'delhi'},
  'id': 'hwrdkam9h',
  'type': 'tool_call'}]

# Tools Execution Loop

In [30]:


# Step 1: Model generation tool calls
messages = [{"role": "user", "content": "What is the weather in delhi"}]
ai_msg= model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with generated arguments
    tool_result= get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step3: Pass the results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


Note: The response 'It's sunny in delhi' is not a function call and I've added it to provide a context to the function call.


In [31]:
messages

[{'role': 'user', 'content': 'What is the weather in delhi'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '25jg9vc5g', 'function': {'arguments': '{"location":"delhi"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 217, 'total_tokens': 232, 'completion_time': 0.027599663, 'completion_tokens_details': None, 'prompt_time': 0.011983543, 'prompt_tokens_details': None, 'queue_time': 0.272650647, 'total_time': 0.039583206}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_8639719ff2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d6c66-f1a5-7510-afa5-524ab723edeb-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'delhi'}, 'id': '25jg9vc5g', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 217, 'output_tokens': 15, 'total_tokens': 232}),
 ToolMessage(content="It's sunny

In [32]:
model_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000240B69FA5A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000240B6AB87D0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'get weather at a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])